# Study-level Evaluator
- **df_metadata** (real-world DICOM metadata)
    - `study_id`: Study identifier
    - `file_id`: DICOM file identifier
    - `IOD`: DICOM Information Object Definition (IOD)
    - `Tag`: DICOM Tag
    - `Value`: Tag value
- **df_standard** (DICOM standard definition)
    - `IOD`: DICOM IOD
    - `Tag`: DICOM Tag
    - `Attribute Name`: DICOM Tag's Attribute Name

In [ ]:
import csv

import numpy as np
import pandas as pd
import os

In [ ]:
import sys
dir_function = '../DicomStandardEvaluator/Evaluator'
sys.path.append(dir_function)

In [ ]:
data_dir = '/nfs/project/WellcomeHDN/kch-mr/metadata/dicom_heterogeneity'
output_dir = '/nfs/project/WellcomeHDN/kch-mr/metadata/dicom_heterogeneity'

## Data Set (DICOM Metadata)
**Columns**:
- **IOD**: IOD Specification according to `(0008,0016) SOP Class UID` ([Table B.5-1.Standard SOP Classes](https://dicom.nema.org/medical/Dicom/2025c/output/chtml/part04/sect_B.5.html))
- **study_id**: unique id per dicom studies 
- **series_id**: unique id per dicom series
- **file_id**: unique id per dicom instances
- **Manufacturer (optional)**: `(0008,0070) Manufacturer` - e.g., 'Siemens Healthineers', 'GE Healthcare',
       'Canon Medical Systems Corporation', 'Philips Healthcare',
       'Hitachi Healthcare Corporation', 'Shimadzu Corporation',
       'AI Lab Co., Ltd.', 'Scimedix Corporation', 'Agfa HealthCare N.V.',
       'Carestream Health, Inc.',
       'Konica Minolta Healthcare Americas, Inc.', None, 'Hologic, Inc.',
       'MEDI-FUTURE, Inc.', 'IMS Giotto S.p.A.', 'FUJIFILM Corporation',
       'GENORAY Co., Ltd.', 'DRTech Corporation'
- **ScannerModel (optional)**: - `(0008,1090) Manufacturer's Model Name` e.g., 'SOMATOM Definition AS', 'LightSpeed16', 'Asteion', 'HiSpeed',
       'SOMATOM Spirit', 'SOMATOM Perspective', 'CT/e', 'LightSpeed',
       'BrightSpeed', 'Brilliance 6', 'Pronto', 'LightSpeed VCT',
       'SOMATOM Definition Flash', 'SOMATOM Definition AS+', 'Aquilion',
       'Brilliance 16', 'SOMATOM Emotion Duo', 'SOMATOM Emotion 6',
       'Optima CT660', 'SOMATOM Emotion 16', 'Alexion', 'Biograph20',
       None, 'Mx8000', 'SOMATOM Volume Zoom', 'Supria',
       'Brilliance iCT 256', 'ProSpeed FII', 'Sytec SRi', 'Brilliance 64',
       'SCT-4800TC', 'AIRIS Vento', 'MAGNETOM Essenza', 'MAGNETOM Avanto',
       'Signa Excite 1.5T', 'Achieva', 'GoldSeal Signa HDxt', 'Ingenia',
       'Intera', 'Genesis Signa', 'AIRIS II', 'MAGNETOM Espree',
       'MagFinder II', 'MAGNETOM Trio', 'Discovery MR750w',
       'Signa profile excite', 'Optima MR430s 1.5T', 'SM160',
       'MAGNETOM Skyra', 'ADC', 'CLASSIC CR', '0862', 'Senographe DS',
       'KODAK DirectView CR 975', 'Lorad Selenia', 'CR 85-X', 'BRESTIGE',
       'CR 75', 'GIOTTO IMAGE MD', 'MAMMOMAT Inspiration',
       'Senographe 2000D', 'KODAK DirectView CR 850',
       'Selenia Dimensions', 'FCR 5000 CR', 'DMX-600', 'RSM 1824C'
- **Tag**
- **AttributeName**
- **Value**

In [ ]:
df_dataset= pd.read_csv(os.path.join(data_dir, 'df_metadata_kch-mr.csv'), dtype=str)

In [ ]:
df_dataset.head()

In [ ]:
def table_1(df):
    summary = pd.DataFrame({
        'n_manufacturer': df.groupby('IOD', dropna=False)['Manufacturer'].nunique(),
        'n_scannermodel': df.groupby('IOD', dropna=False)['ScannerModel'].nunique(),
        'n_study_global': df.groupby('IOD', dropna=False)['study_id'].nunique(),
        'n_series_global': df.groupby('IOD', dropna=False)['series_id'].nunique(),
        'n_file_global': df.groupby('IOD', dropna=False)['file_id'].nunique(),
        'n_tag': df.groupby('IOD', dropna=False)['Tag'].size(),
        'n_tag/n_file': df.groupby('IOD', dropna=False)['Tag'].size()/df.groupby('IOD', dropna=False)['file_id'].nunique()
    })
    return summary

summary = table_1(df_dataset)
summary.to_csv(os.path.join(output_dir, 'summary_counts.csv'))
summary

## Reference Set (DICOM Standard 2025c)
- Mandatory Modality-specific Modules

In [ ]:
# 2025c standard
df_standard = pd.read_excel("../files/DicomStandardReference_2025c/C2025MandatoryModalityspecificModules_ReferenceSet.xlsx")
print(f"2025c standard: {len(df_standard)} rows with {df_standard['Tag'].nunique()} unique tags")

In [ ]:
display(df_standard.groupby('IOD')['Tag'].nunique())
display(df_standard.head(1))

## Evaluation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import tabulate
import sys

sys.path.append('Evaluator')
from DicomCodeStandardEvaluator import DicomCodeStandardEvaluator

In [ ]:
#df_standard = df_standard.copy()
#df_dataset = df_dataset.copy() 

print(f'Standard: {df_standard.shape[0]} rows with {df_standard['Tag'].nunique()} unique tags')
print(f'Dataset: {df_dataset.shape[0]} rows with {df_dataset['Tag'].nunique()} unique tags')

### Completeness (Study-level)

In [ ]:
from DicomCodeStandardEvaluator_withoutVR import DicomCodeStandardEvaluator_withoutVR # Completeness Only (No need for 'VR')
evaluator_2025c = DicomCodeStandardEvaluator_withoutVR(df_dataset, df_standard)

In [ ]:
study_rates_2025c, study_stats_2025c = evaluator_2025c.analyze_rates_with_stats(group_cols=['IOD', 'study_id', 'Manufacturer', 'ScannerModel'])

#### study_rates_2025c

In [ ]:
# Information Completeness Index (ICI) = tag existence x value existence
study_rates_2025c['ICI'] = study_rates_2025c['tag_existence_rate'] * study_rates_2025c['value_existence_rate']

# Manufacturer, ScannerModel
iod_study_rates_2025c = pd.merge(study_rates_2025c, df_dataset[['IOD', 'study_id', 'Manufacturer', 'ScannerModel']].drop_duplicates(keep='first'), on = ['IOD', 'study_id'], how='left')
iod_study_rates_2025c.head()

In [ ]:
# SAVE
import csv
try:
    study_rates_2025c.to_excel(os.path.join(output_dir, 'study_rates_2025c.xlsx'), index=False)
    iod_study_rates_2025c.to_excel(os.path.join(output_dir, 'iod_study_rates_2025c.xlsx'), index=False)
except Exception as e:
    print(f"Error saving to Excel: {e}. Trying CSV.")
    study_rates_2025c.to_csv(os.path.join(output_dir, 'study_rates_2025c.csv'), index=False, quoting=csv.QUOTE_NONNUMERIC)
    iod_study_rates_2025c.to_csv(os.path.join(output_dir, 'iod_study_rates_2025c.csv'), index=False, quoting=csv.QUOTE_NONNUMERIC)

In [ ]:
# Try summarising by Manufacturer and ScannerModel
# Count
manufacturer_n = study_rates_2025c.drop_duplicates(subset='study_id').groupby(['IOD', 'Manufacturer', 'ScannerModel']).agg({
    'study_id': ['count'],
}).reset_index()
manufacturer_n.columns = ['IOD', 'Manufacturer', 'ScannerModel', 'n_studies']
manufacturer_n.head()

In [ ]:
# Only use tag existence rate, because in this dataset all tags that exist have values (value existence is only 0 or 1)
study_rates_2025c['Make Model'] = study_rates_2025c['Manufacturer'] + ' ' + study_rates_2025c['ScannerModel']
manufacturer_exist = study_rates_2025c.groupby(['IOD', 'Tag', 'Attribute Name', 'Make Model']).agg({
    'tag_existence_rate': ['mean', 'std'],
}).reset_index()
manufacturer_exist.fillna(0, inplace=True)  # Set SD for groups with only one study to 0 instead of NaN to avoid it mucking up aggregation later
manufacturer_exist.head()

In [ ]:
# Pivot
manufacturer_exist.columns = ['IOD', 'Tag', 'Attribute Name', 'Make Model', 'mean', 'std']
manufacturer_exist_mean = manufacturer_exist.pivot_table(index=['IOD', 'Tag', 'Attribute Name'], columns=['Make Model'], values=['mean'], aggfunc='sum').reset_index()
manufacturer_exist_std = manufacturer_exist.pivot_table(index=['IOD', 'Tag', 'Attribute Name'], columns=['Make Model'], values=['std'], aggfunc='sum').reset_index()
manufacturer_exist_mean.columns = [' '.join(col).strip().replace('mean ', '') for col in manufacturer_exist_mean.columns]
manufacturer_exist_std.columns = [' '.join(col).strip().replace('std ', '') for col in manufacturer_exist_std.columns]
manufacturer_exist_mean.head()

In [ ]:
# Save
with pd.ExcelWriter(os.path.join(output_dir, 'manufacturer_stats_2025c.xlsx')) as xl:
    manufacturer_n.to_excel(xl, sheet_name='N', index=False)
    manufacturer_exist_mean.to_excel(xl, sheet_name='tag_existence_mean', index=False)
    manufacturer_exist_std.to_excel(xl, sheet_name='tag_existence_std', index=False)

#### study_stats_2025c

In [ ]:
print(study_stats_2025c.shape)
study_stats_2025c.head()

In [ ]:
# 1. Convert to pivot table (create wide-format columns for each metric and statistic)
study_stats_2025c_wide = study_stats_2025c.pivot_table(
    index=['IOD', 'Tag', 'n_groups'],
    columns='Metric',
    values=['Mean', 'Std']
).reset_index()

# 2. Clean up column names (MultiIndex → single column)
study_stats_2025c_wide.columns = ['_'.join(col).strip('_') for col in study_stats_2025c_wide.columns.values]

# 3. Reorder columns
cols = ['IOD', 'Tag'] + \
       [col for col in study_stats_2025c_wide.columns if 'tag_existence_rate' in col] + \
       [col for col in study_stats_2025c_wide.columns if 'value_existence_rate' in col] + \
       [col for col in study_stats_2025c_wide.columns if 'value_standardization_rate' in col] + \
       [col for col in study_stats_2025c_wide.columns if 'value_diversity' in col] + \
       ['n_groups']
study_stats_2025c_wide = study_stats_2025c_wide[cols]
study_stats_2025c_wide.head(2)

In [ ]:
# Merge with df_standard
iod_study_stats_2025c = pd.merge(study_stats_2025c_wide, df_standard, on=['IOD', 'Tag'], how='left')
iod_study_stats_2025c = iod_study_stats_2025c[['IOD', 'Tag', 'Attribute Name', 'Type', 'Type_Group', 
       'Attribute Description', 'Source', 'n_groups', 
       'Mean_tag_existence_rate', 'Std_tag_existence_rate',
       'Mean_value_existence_rate', 'Std_value_existence_rate'
       ]]
iod_study_stats_2025c.head()

In [ ]:
# SAVE
study_stats_2025c.to_excel(os.path.join(output_dir, 'study_stats_2025c.xlsx'), index=False)
iod_study_stats_2025c.to_excel(os.path.join(output_dir, 'iod_study_stats_2025c.xlsx'), index=False)